[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C27_Model_Compression_Course/02_gptq_awq/02_gptq_awq.ipynb)

# 02 · GPTQ 与 AWQ（用 numpy 从零实现）

把 int4 从「明显掉点」做到「近乎无损」的两条 PTQ 主线，**从零写、与 RTN 对比**。

**路线**：
1. 搭建场景：层 W + 相关性校准激活 X；RTN 作基线
2. 为什么 RTN 不是最优：输出误差 vs 权重误差
3. **GPTQ**：逆 Hessian 逐列误差补偿 → MSE 压到 RTN 的 ~一半
4. 阻尼的必要性：病态 Hessian 与数值稳定
5. **AWQ**：激活感知缩放 + α 搜索 → 保护显著通道
6. 三方对比：RTN vs GPTQ vs AWQ
7. ✏️ 练习（OBQ 单步更新 / AWQ scale / 显著通道 / 对比 RTN）
8. 📖 答案 · 🧪 真实 GPT-2 权重胶囊

> **本课纪律**：每个算法都和 RTN/全精度对拍。GPTQ/AWQ 的层输出 MSE 必须 < RTN，才算实现正确。

## 1 · 搭场景：层 W、相关性激活 X、RTN 基线

真实激活**通道间有相关性**（不是独立噪声），这正是 GPTQ 的逆 Hessian 能发挥作用的前提。
我们造一个有协方差结构的 X，并实现 per-row absmax int4 的 RTN 作为基线。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

d_out, d_in, n = 16, 64, 256
# 相关性激活：X 有协方差结构（真实激活的特征）
A = rng.standard_normal((d_in, d_in))
cov_sqrt = A @ A.T / d_in
X = cov_sqrt @ rng.standard_normal((d_in, n))     # (d_in, n) 通道间相关
W = rng.standard_normal((d_out, d_in)) * 0.1      # 层权重

def out_mse(W, Wq, X):
    return np.mean((W @ X - Wq @ X) ** 2)         # 层输出 MSE —— 我们真正在意的

def rtn(W, bits=4):
    '''per-row absmax int4 就近舍入(基线)。'''
    qmax = 2**(bits-1) - 1
    s = np.abs(W).max(axis=1, keepdims=True) / qmax
    return s * np.round(W / s).clip(-qmax, qmax)

rtn_mse = out_mse(W, rtn(W, 4), X)
print(f'RTN int4 层输出 MSE = {rtn_mse:.4e}')
print(f'激活相关性(off-diag 占比): {np.abs(X@X.T - np.diag(np.diag(X@X.T))).mean() / np.abs(X@X.T).mean():.2f}')
assert rtn_mse > 0
print('✅ 场景就绪：相关性激活 + RTN 基线。下面看 GPTQ/AWQ 怎么打败它')

## 2 · 为什么 RTN 不是最优：输出误差 ≠ 权重误差

RTN 最小化 `‖W-Ŵ‖`（权重误差），但我们要的是 `‖WX-ŴX‖`（输出误差）。
证据：人为构造一个**权重误差更大、但输出误差更小**的量化，说明「权重最近」不等于「输出最好」。

In [ ]:
Wq_rtn = rtn(W, 4)
w_err_rtn = np.linalg.norm(W - Wq_rtn)
o_err_rtn = np.linalg.norm(W@X - Wq_rtn@X)

# 在 RTN 基础上，沿『对输出影响小』的方向加一点权重扰动:
# 用激活协方差的最小特征向量方向(输出几乎不变, 但权重变了)
Hk = X @ X.T
evals, evecs = np.linalg.eigh(Hk)
low_dir = evecs[:, 0]                              # 最小特征值方向(对输出影响最小)
Wq_alt = Wq_rtn + 0.02 * np.outer(rng.standard_normal(d_out), low_dir)
w_err_alt = np.linalg.norm(W - Wq_alt)
o_err_alt = np.linalg.norm(W@X - Wq_alt@X)

print(f'RTN   : 权重误差={w_err_rtn:.4f}  输出误差={o_err_rtn:.4f}')
print(f'扰动版: 权重误差={w_err_alt:.4f}  输出误差={o_err_alt:.4f}')
assert w_err_alt > w_err_rtn, '扰动版权重误差更大'
print('\n沿低影响方向扰动: 权重误差变大, 但输出误差几乎不变 ->')
print('✅ 说明「权重最近」≠「输出最好」: 存在权重更偏、但输出一样好的解。GPTQ 就去找这种解')

## 3 · GPTQ：逆 Hessian 逐列误差补偿

核心循环：逐列量化，每列产生误差 `e=(w-q)/Hinv[j,j]`，把 `e` 按逆 Hessian 补偿到右侧未量化列。
Hessian `H=XXᵀ`（+阻尼），所有行共享同一个 `Hinv`（因为 H 只依赖 X）。

In [ ]:
def gptq(W, X, bits=4, damp=1e-2):
    d_out, d_in = W.shape
    qmax = 2**(bits-1) - 1
    H = X @ X.T                                    # (d_in,d_in) 层重建损失的 Hessian
    H = H + damp * np.mean(np.diag(H)) * np.eye(d_in)   # 阻尼: 保证可逆+数值稳定
    Hinv = np.linalg.inv(H)
    Hd = np.diag(Hinv).copy()
    s = np.abs(W).max(axis=1, keepdims=True) / qmax     # per-row scale
    Wq = W.copy().astype(float)
    for j in range(d_in):                          # 逐列
        w_col = Wq[:, j]
        q_col = s[:, 0] * np.round(w_col / s[:, 0]).clip(-qmax, qmax)   # 量化这一列
        err = (w_col - q_col) / Hd[j]              # 归一化误差
        Wq[:, j] = q_col
        if j + 1 < d_in:                           # 把误差补偿到右侧未量化列
            Wq[:, j+1:] -= np.outer(err, Hinv[j, j+1:])
    return Wq

gptq_mse = out_mse(W, gptq(W, X, 4), X)
print(f'RTN  int4 输出 MSE = {rtn_mse:.4e}')
print(f'GPTQ int4 输出 MSE = {gptq_mse:.4e}')
print(f'GPTQ/RTN = {gptq_mse/rtn_mse:.3f}  (越小越好)')
assert gptq_mse < rtn_mse, 'GPTQ 应当优于 RTN'
print('✅ GPTQ 把层输出 MSE 压到 RTN 的约一半 —— 逆 Hessian 误差补偿生效')

## 4 · 阻尼的必要性：病态 Hessian

校准样本少时 `XXᵀ` 可能**病态/奇异**，直接求逆会数值爆炸或 `LinAlgError`。
阻尼 `H += λ·mean(diag)·I` 给 Hessian 一点正则。我们用**样本数 < 维度**制造病态，看阻尼救场。

In [ ]:
# 病态场景: 样本数 n_small < d_in -> XXᵀ 秩亏(奇异)
n_small = 32                                        # < d_in=64
X_small = cov_sqrt @ rng.standard_normal((d_in, n_small))
H_bad = X_small @ X_small.T
print(f'病态 Hessian: 条件数 = {np.linalg.cond(H_bad):.2e} (极大->病态)')

# 无阻尼: 条件数爆炸(可能 inf); 有阻尼: 受控
damp = 1e-2
H_damped = H_bad + damp * np.mean(np.diag(H_bad)) * np.eye(d_in)
print(f'加阻尼后: 条件数 = {np.linalg.cond(H_damped):.2e} (受控)')

# GPTQ 在病态场景下: 加阻尼仍能跑出 < RTN 的结果
gptq_small = out_mse(W, gptq(W, X_small, 4, damp=1e-2), X_small)
rtn_small = out_mse(W, rtn(W, 4), X_small)
print(f'病态场景: RTN MSE={rtn_small:.4e}  GPTQ(阻尼) MSE={gptq_small:.4e}')
assert np.linalg.cond(H_damped) < np.linalg.cond(H_bad)
assert np.isfinite(gptq_small), '加阻尼后 GPTQ 应数值稳定'
print('✅ 阻尼把病态 Hessian 的条件数压下来，GPTQ 才能数值稳定地跑')

## 5 · AWQ：激活感知缩放 + α 搜索

给显著(大激活)通道的权重乘 `s_j=(act_j/mean)^α`、量化后除回，等价不变但保护重要权重。
在 α∈{0,0.25,0.5,0.75,1} 里搜使输出 MSE 最小的（α=0 即退化成 RTN）。

In [ ]:
act_scale = np.abs(X).mean(axis=1)                 # (d_in,) 每个输入通道的激活幅度

def awq(W, X, bits=4, alpha=0.5):
    qmax = 2**(bits-1) - 1
    sj = act_scale ** alpha                         # 按激活幅度的幂次设缩放
    sj = sj / sj.mean()                             # 归一化(避免整体缩放)
    Ws = W * sj[None, :]                            # 放大显著通道的权重
    srow = np.abs(Ws).max(axis=1, keepdims=True) / qmax
    Wsq = srow * np.round(Ws / srow).clip(-qmax, qmax)
    return Wsq / sj[None, :]                        # 除回(数学等价)

print('AWQ α 搜索:')
best_alpha, best_mse = None, np.inf
for alpha in [0, 0.25, 0.5, 0.75, 1.0]:
    m = out_mse(W, awq(W, X, 4, alpha), X)
    flag = ' <- RTN(α=0)' if alpha == 0 else ''
    print(f'  α={alpha:.2f}: 输出 MSE = {m:.4e}{flag}')
    if m < best_mse: best_alpha, best_mse = alpha, m
print(f'最优 α = {best_alpha} (MSE={best_mse:.4e})')
assert best_mse < out_mse(W, awq(W, X, 4, 0.0), X) + 1e-12, 'AWQ 最优应不差于 α=0(RTN)'
assert best_alpha > 0, 'AWQ 应找到比 RTN 更好的非零 α'
print('✅ AWQ 搜到非零 α，保护显著通道，MSE 优于 RTN')

## 6 · 三方对比：RTN vs GPTQ vs AWQ

把三者放一起，在同一个 (W, X) 上比 int4 层输出 MSE。结论：GPTQ、AWQ 都明显优于 RTN。

In [ ]:
def awq_best(W, X, bits=4):
    return min((out_mse(W, awq(W, X, bits, a), X), a) for a in [0,0.25,0.5,0.75,1.0])[0]

results = {
    'RTN':  out_mse(W, rtn(W, 4), X),
    'GPTQ': out_mse(W, gptq(W, X, 4), X),
    'AWQ':  awq_best(W, X, 4),
}
base = results['RTN']
print(f"{'方法':<6}{'int4 输出 MSE':>16}{'相对 RTN':>12}")
for k, v in results.items():
    print(f'{k:<6}{v:>16.4e}{v/base:>11.1%}')
assert results['GPTQ'] < base and results['AWQ'] < base, 'GPTQ/AWQ 都应优于 RTN'
print('\n✅ GPTQ 与 AWQ 都明显打败 RTN —— 把激活纳入考量是 int4 近乎无损的关键')

---
## ✏️ 练习 1：OBQ 单列更新

实现 GPTQ 的一步核心更新 `obq_step(Wq, Hinv, j, s, qmax)`：量化第 `j` 列、算归一化误差、补偿到右侧列。
返回更新后的 `Wq`。这是 GPTQ 的原子操作。

In [ ]:
def obq_step(Wq, Hinv, j, s, qmax):
    '''量化 Wq 的第 j 列, 把误差按逆 Hessian 补偿到 j 右侧的列。原地修改并返回 Wq。'''
    # TODO:
    #   w_col = Wq[:, j]
    #   q_col = s[:,0] * round(w_col/s[:,0]) 再 clip 到 [-qmax,qmax]
    #   err = (w_col - q_col) / Hinv[j,j]
    #   Wq[:, j] = q_col
    #   若 j+1 < d_in:  Wq[:, j+1:] -= outer(err, Hinv[j, j+1:])
    #   返回 Wq
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
qmax = 7
H = X@X.T; H = H + 1e-2*np.mean(np.diag(H))*np.eye(d_in); Hinv = np.linalg.inv(H)
s = np.abs(W).max(axis=1, keepdims=True) / qmax
Wq = W.copy().astype(float)
for j in range(d_in):
    Wq = obq_step(Wq, Hinv, j, s, qmax)
mse = out_mse(W, Wq, X)
assert mse < rtn_mse, f'逐列 OBQ 应优于 RTN: {mse:.3e} vs {rtn_mse:.3e}'
# 量化后每个值应落在网格上
assert np.all(np.abs(np.round(Wq/s) - Wq/s) < 1e-6), '最终权重应在量化网格上'
print(f'✅ 练习 1 通过：逐列 OBQ 输出 MSE={mse:.3e} < RTN {rtn_mse:.3e}')

## ✏️ 练习 2：AWQ 缩放

实现 `awq_scale(W, act_scale, alpha, bits)`：按激活幅度的 α 次幂给输入通道缩放、量化、除回。
目标：α>0 时输出 MSE 应不差于 α=0（RTN）。

In [ ]:
def awq_scale(W, act_scale, alpha=0.5, bits=4):
    # TODO:
    #   qmax = 2**(bits-1)-1
    #   sj = act_scale**alpha; sj = sj/sj.mean()
    #   Ws = W * sj[None,:]
    #   per-row absmax 量化 Ws -> Wsq
    #   返回 Wsq / sj[None,:]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
m0 = out_mse(W, awq_scale(W, act_scale, 0.0, 4), X)   # α=0 即 RTN
m_best = min(out_mse(W, awq_scale(W, act_scale, a, 4), X) for a in [0.25,0.5,0.75,1.0])
assert abs(m0 - rtn_mse) / rtn_mse < 0.05, 'α=0 应约等于 RTN'
assert m_best < m0, 'AWQ 应找到优于 RTN 的 α'
print(f'✅ 练习 2 通过：AWQ 最优 MSE={m_best:.3e} < RTN(α=0) {m0:.3e}')

## ✏️ 练习 3：找出显著通道

实现 `salient_channels(X, top_frac)`：返回激活幅度最大的前 `top_frac` 比例输入通道的下标。
这些就是 AWQ 要重点保护的通道。

In [ ]:
def salient_channels(X, top_frac=0.01):
    # TODO: 按每通道激活幅度 mean(|X[i]|) 排序, 返回最大的 top_frac 比例通道的下标(数组)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 人为把某些通道激活放大, 它们应被识别为显著
X2 = X.copy()
boost = [3, 17, 40]
X2[boost] *= 10.0
act2 = np.abs(X2).mean(axis=1)
sal = salient_channels(X2, top_frac=5/d_in)           # 取前 5 个
assert set(boost).issubset(set(sal.tolist())), '放大的通道应被识别为显著'
assert len(sal) == 5
print(f'✅ 练习 3 通过：识别出显著通道 {sorted(sal.tolist())} (含人为放大的 {boost})')

## ✏️ 练习 4：GPTQ vs AWQ vs RTN 全对比

实现 `compare_methods(W, X, bits)`：返回字典 `{'RTN':mse, 'GPTQ':mse, 'AWQ':mse}`。
目标：GPTQ 和 AWQ 都 < RTN。

In [ ]:
def compare_methods(W, X, bits=4):
    # TODO: 用本 notebook 的 rtn / gptq / awq 算三者的 out_mse, 返回字典
    #       AWQ 取 α∈{0,0.25,0.5,0.75,1.0} 的最优
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
res = compare_methods(W, X, 4)
assert set(res.keys()) == {'RTN', 'GPTQ', 'AWQ'}
assert res['GPTQ'] < res['RTN'] and res['AWQ'] < res['RTN']
for k in ['RTN', 'GPTQ', 'AWQ']:
    print(f"  {k}: {res[k]:.4e}  ({res[k]/res['RTN']:.0%} of RTN)")
print('✅ 练习 4 通过：GPTQ 与 AWQ 都打败 RTN')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def obq_step(Wq, Hinv, j, s, qmax):
    w_col = Wq[:, j]
    q_col = s[:, 0] * np.round(w_col / s[:, 0]).clip(-qmax, qmax)
    err = (w_col - q_col) / Hinv[j, j]
    Wq[:, j] = q_col
    if j + 1 < Wq.shape[1]:
        Wq[:, j+1:] -= np.outer(err, Hinv[j, j+1:])
    return Wq

In [ ]:
# 练习 2 参考答案
def awq_scale(W, act_scale, alpha=0.5, bits=4):
    qmax = 2**(bits-1) - 1
    sj = act_scale ** alpha; sj = sj / sj.mean()
    Ws = W * sj[None, :]
    srow = np.abs(Ws).max(axis=1, keepdims=True) / qmax
    Wsq = srow * np.round(Ws / srow).clip(-qmax, qmax)
    return Wsq / sj[None, :]

In [ ]:
# 练习 3 参考答案
def salient_channels(X, top_frac=0.01):
    mag = np.abs(X).mean(axis=1)
    k = max(1, int(round(top_frac * X.shape[0])))
    return np.argsort(mag)[::-1][:k]

In [ ]:
# 练习 4 参考答案
def compare_methods(W, X, bits=4):
    act = np.abs(X).mean(axis=1)
    def awq_b(W, X):
        return min(out_mse(W, awq_scale(W, act, a, bits), X) for a in [0,0.25,0.5,0.75,1.0])
    return {'RTN': out_mse(W, rtn(W, bits), X),
            'GPTQ': out_mse(W, gptq(W, X, bits), X),
            'AWQ': awq_b(W, X)}

---
## 🧪 真实数据胶囊：在真实 GPT-2 权重上比 GPTQ vs RTN

用**真实 GPT-2** 的一个权重张量 + 合成相关性激活，验证 GPTQ 在真实权重上也优于 RTN。
**联网失败自动回退**到统计匹配的合成权重，结论不变。

In [ ]:
def load_gpt2_attn_weight():
    '''真实 GPT-2 的一个注意力投影权重; 失败回退合成。返回 (d_out, d_in) 的小切片。'''
    try:
        from transformers import GPT2Model
        m = GPT2Model.from_pretrained('gpt2')
        W = m.h[0].attn.c_proj.weight.detach().numpy().astype(np.float64)
        W = W[:32, :64]                              # 取小切片便于 CPU 演示
        print(f'[真实 GPT-2] c_proj 切片 shape={W.shape}')
        return W
    except Exception as e:
        print(f'[回退合成] ({type(e).__name__}) 用统计匹配合成权重')
        rng2 = np.random.default_rng(7)
        return rng2.standard_normal((32, 64)) * 0.12

Wg = load_gpt2_attn_weight()
dout2, din2 = Wg.shape
Ag = rng.standard_normal((din2, din2)); covg = Ag@Ag.T/din2
Xg = covg @ rng.standard_normal((din2, 200))         # 相关性激活
print(f'权重统计: std={Wg.std():.4f}, absmax={np.abs(Wg).max():.4f}')

In [ ]:
def gptq_beats_rtn(W, X, bits=4):
    # TODO: 返回 (rtn_mse, gptq_mse), 用本 notebook 的 rtn/gptq/out_mse
    raise NotImplementedError

In [ ]:
# 自测
r_mse, g_mse = gptq_beats_rtn(Wg, Xg, 4)
print(f'真实(或合成)GPT-2 权重 int4:')
print(f'  RTN  输出 MSE = {r_mse:.4e}')
print(f'  GPTQ 输出 MSE = {g_mse:.4e}  ({g_mse/r_mse:.0%} of RTN)')
assert g_mse < r_mse, f'GPTQ 应优于 RTN: {g_mse:.3e} vs {r_mse:.3e}'
print('✅ 胶囊通过：真实权重上 GPTQ 仍优于 RTN —— 逆 Hessian 补偿在真实分布上同样有效')

In [ ]:
# 📖 胶囊参考答案
def gptq_beats_rtn(W, X, bits=4):
    return out_mse(W, rtn(W, bits), X), out_mse(W, gptq(W, X, bits), X)

### 小结
- **RTN 不是最优**：它最小化 `‖W-Ŵ‖`(权重误差)，但我们要 `‖WX-ŴX‖`(输出误差)。差别在激活 X。
- **GPTQ** = 逐列量化 + 逆 Hessian 误差补偿。`H=XXᵀ`(+阻尼)，所有行共享 `Hinv`，把每列舍入误差摊到右侧未量化列 → 输出 MSE ~RTN 的一半。
- **阻尼必需**：校准样本少时 Hessian 病态，`H+=λ·mean(diag)·I` 保数值稳定。
- **AWQ** = 给显著(大激活)通道乘保护缩放 `s_j=(act/mean)^α`、量化后除回(等价不变)，搜最优 α。比 GPTQ 轻、快、无额外反量化开销。
- **等价缩放变换** `Y=(WP)(P⁻¹X)` 是一大类技巧的共性：AWQ/SmoothQuant/QuaRot 都在选好的 P 重新分配量化误差负担。
- 三者都可叠加 **group-wise**(模块01)。生产里 GPTQ/AWQ 常一起评测、各有胜负，但都远胜 RTN。

下一站：**模块 03 · fp8 训练** —— 换个战场，看低精度浮点怎么用于训练而不 NaN。